# Docling PDF pipeline test

End-to-end Docling PDF pipeline that extracts **text**, **tables**, **images** and runs **OCR**, then:

1. prints the exported Markdown,
2. renders every extracted picture inline,
3. renders every extracted table as a DataFrame plus its cropped image,
4. writes Markdown (embedded and referenced), HTML and PNG assets to disk.

## Requirements

```powershell
.venv\Scripts\python.exe -m pip install docling
```

`pandas`, `pillow`, `ipython` and `ipykernel` are already in this venv and cover the DataFrame and image display below.

The default OCR engine (EasyOCR) downloads its models on first run, so the first conversion is slow.

## How the pipeline is wired

| Option | Effect |
| --- | --- |
| `do_ocr` | Runs OCR over bitmap regions (or the whole page, see `FORCE_FULL_PAGE_OCR`) |
| `ocr_options` | Picks the OCR engine and languages (EasyOCR / Tesseract / RapidOCR) |
| `do_table_structure` | Runs the TableFormer model so tables come out as real cell grids |
| `table_structure_options.do_cell_matching` | Maps predicted cells back to the PDF text cells instead of OCR-ing them |
| `generate_page_images` | Keeps a rendered bitmap of each page (needed to crop table images) |
| `generate_picture_images` | Keeps a cropped bitmap for each detected figure |
| `images_scale` | Render resolution multiplier, `1.0` is roughly 72 DPI |

In [ ]:
from __future__ import annotations

import logging
import time
from pathlib import Path

from IPython.display import Markdown, display

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    EasyOcrOptions,
    PdfPipelineOptions,
    TableFormerMode,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode, PictureItem, TableItem

# Accelerator options moved module between Docling releases, so try both homes.
try:
    from docling.datamodel.accelerator_options import (
        AcceleratorDevice,
        AcceleratorOptions,
    )
except ImportError:  # older docling
    from docling.datamodel.pipeline_options import (  # type: ignore[no-redef]
        AcceleratorDevice,
        AcceleratorOptions,
    )

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("docling").setLevel(logging.INFO)

print("docling imports ok")

## Configuration

Change `PDF_PATH` to point at whichever document you want to run. It defaults to the first PDF in `src/tests/fixtures/`.

In [ ]:
def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until we find the folder that contains `src/`."""
    for candidate in [start, *start.parents]:
        if (candidate / "src").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd().resolve())
FIXTURE_DIR = REPO_ROOT / "src" / "tests" / "fixtures"
OUTPUT_DIR = REPO_ROOT / "output" / "docling"

_fixtures = sorted(FIXTURE_DIR.glob("*.pdf"))
PDF_PATH = _fixtures[0] if _fixtures else None

# --- pipeline knobs -------------------------------------------------------
IMAGE_RESOLUTION_SCALE = 2.0     # 1.0 is ~72 DPI, 2.0 is ~144 DPI
OCR_LANGS = ["en"]               # EasyOCR language codes
FORCE_FULL_PAGE_OCR = False      # True for scanned PDFs with no text layer
TABLE_MODE = TableFormerMode.ACCURATE  # or TableFormerMode.FAST
NUM_THREADS = 8
PAGE_RANGE = None                # e.g. (1, 6) to limit pages while testing

# --- display knobs --------------------------------------------------------
MARKDOWN_CHAR_LIMIT = 0          # 0 prints the whole Markdown export
MAX_PICTURES_TO_SHOW = 20
MAX_TABLES_TO_SHOW = 20
SHOW_PAGE_IMAGES = False         # flip on to eyeball what OCR actually saw

print(f"repo root : {REPO_ROOT}")
print(f"pdf       : {PDF_PATH}")
print(f"output    : {OUTPUT_DIR}")
print(f"fixtures  : {len(_fixtures)} PDFs available")
for f in _fixtures:
    print(f"  - {f.name}")

## Build the converter

One `DocumentConverter` configured for `InputFormat.PDF`. Building it is cheap; the models load lazily on first conversion, so reuse the same converter across documents.

In [ ]:
def build_pdf_converter(
    *,
    images_scale: float = IMAGE_RESOLUTION_SCALE,
    ocr_langs: list[str] | None = None,
    force_full_page_ocr: bool = FORCE_FULL_PAGE_OCR,
    table_mode: TableFormerMode = TABLE_MODE,
    num_threads: int = NUM_THREADS,
) -> DocumentConverter:
    """Return a DocumentConverter doing OCR, table structure and image extraction."""
    pipeline_options = PdfPipelineOptions()

    # Text and OCR. `force_full_page_ocr` re-reads the entire page through OCR
    # instead of only the bitmap regions: slower, but the right call for scans.
    pipeline_options.do_ocr = True
    pipeline_options.ocr_options = EasyOcrOptions(
        lang=ocr_langs or OCR_LANGS,
        force_full_page_ocr=force_full_page_ocr,
    )

    # Tables. do_cell_matching maps TableFormer's predicted cells back onto the
    # PDF's own text cells, which keeps the original characters rather than OCR.
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options = TableStructureOptions(
        do_cell_matching=True,
        mode=table_mode,
    )

    # Images. Page images are required for cropping table images later on.
    pipeline_options.images_scale = images_scale
    pipeline_options.generate_page_images = True
    pipeline_options.generate_picture_images = True

    pipeline_options.accelerator_options = AcceleratorOptions(
        num_threads=num_threads,
        device=AcceleratorDevice.AUTO,
    )

    # Optional enrichments. They pull extra models, so they stay off by default.
    # pipeline_options.do_formula_enrichment = True
    # pipeline_options.do_code_enrichment = True
    # pipeline_options.do_picture_classification = True

    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )


converter = build_pdf_converter()
print("converter ready")

## Convert

The first run downloads the layout, TableFormer and EasyOCR models, which can take several minutes and needs network access. Subsequent runs read them from the local cache.

In [ ]:
if PDF_PATH is None or not PDF_PATH.exists():
    raise FileNotFoundError(f"No PDF to convert. PDF_PATH={PDF_PATH}")

convert_kwargs = {}
if PAGE_RANGE is not None:
    convert_kwargs["page_range"] = PAGE_RANGE

start = time.perf_counter()
result = converter.convert(PDF_PATH, **convert_kwargs)
elapsed = time.perf_counter() - start

doc = result.document
doc_stem = result.input.file.stem

print(f"status   : {result.status}")
print(f"elapsed  : {elapsed:.1f}s")
print(f"pages    : {len(doc.pages)}")
if getattr(result, "errors", None):
    for err in result.errors:
        print(f"error    : {err}")

In [ ]:
# What did the pipeline actually find?
pictures = [el for el, _ in doc.iterate_items() if isinstance(el, PictureItem)]
tables = [el for el, _ in doc.iterate_items() if isinstance(el, TableItem)]

print(f"text items : {len(doc.texts)}")
print(f"tables     : {len(tables)}")
print(f"pictures   : {len(pictures)}")

# Breakdown of text items by semantic label (title, section_header, paragraph, ...).
counts: dict[str, int] = {}
for item in doc.texts:
    label = str(getattr(item, "label", "unknown"))
    counts[label] = counts.get(label, 0) + 1

print("\ntext items by label:")
for label, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {label:<20} {n}")

## Markdown output

First as raw text (so you can see exactly what Docling emits, including the table pipes and image placeholders), then rendered.

In [ ]:
markdown = doc.export_to_markdown()

print(f"markdown length: {len(markdown):,} chars\n")
print("=" * 100)
if MARKDOWN_CHAR_LIMIT and len(markdown) > MARKDOWN_CHAR_LIMIT:
    print(markdown[:MARKDOWN_CHAR_LIMIT])
    print(f"\n... truncated, {len(markdown) - MARKDOWN_CHAR_LIMIT:,} chars remaining ...")
else:
    print(markdown)
print("=" * 100)

In [ ]:
# Rendered view. ImageRefMode.EMBEDDED inlines every figure as a base64 data URI,
# so the pictures show up in the notebook rather than as broken links.
markdown_embedded = doc.export_to_markdown(image_mode=ImageRefMode.EMBEDDED)
display(Markdown(markdown_embedded))

## Extracted images

Every `PictureItem` with its caption, provenance page and cropped bitmap.

In [ ]:
def page_of(element) -> str:
    prov = getattr(element, "prov", None)
    if prov:
        return str(prov[0].page_no)
    return "?"


if not pictures:
    print("No pictures detected in this document.")

for i, picture in enumerate(pictures[:MAX_PICTURES_TO_SHOW], start=1):
    caption = picture.caption_text(doc) or "(no caption)"
    print(f"\n--- picture {i}/{len(pictures)} | page {page_of(picture)} | {picture.self_ref}")
    print(f"caption: {caption}")

    # Any classification / description annotations the enrichment models added.
    for annotation in getattr(picture, "annotations", []) or []:
        print(f"annotation: {annotation}")

    image = picture.get_image(doc)
    if image is None:
        print("no bitmap available (is generate_picture_images enabled?)")
        continue
    print(f"size: {image.width}x{image.height}")
    display(image)

if len(pictures) > MAX_PICTURES_TO_SHOW:
    print(f"\n... {len(pictures) - MAX_PICTURES_TO_SHOW} more pictures not shown")

## Extracted tables

Each table is shown three ways: as a pandas DataFrame (the structured result of TableFormer), as Markdown, and as the cropped bitmap so you can check the parse against the page.

In [ ]:
def table_markdown(table) -> str:
    """TableItem.export_to_markdown gained a `doc` argument in newer docling-core."""
    try:
        return table.export_to_markdown(doc)
    except TypeError:
        return table.export_to_markdown()


if not tables:
    print("No tables detected in this document.")

dataframes = []
for i, table in enumerate(tables[:MAX_TABLES_TO_SHOW], start=1):
    caption = table.caption_text(doc) or "(no caption)"
    print(f"\n--- table {i}/{len(tables)} | page {page_of(table)} | {table.self_ref}")
    print(f"caption: {caption}")

    try:
        df = table.export_to_dataframe()
        dataframes.append(df)
        print(f"shape: {df.shape[0]} rows x {df.shape[1]} cols")
        display(df)
    except Exception as exc:  # a table the model could not resolve into a grid
        print(f"dataframe export failed: {exc}")

    display(Markdown(table_markdown(table)))

    image = table.get_image(doc)
    if image is not None:
        display(image)

if len(tables) > MAX_TABLES_TO_SHOW:
    print(f"\n... {len(tables) - MAX_TABLES_TO_SHOW} more tables not shown")

## Page images (optional)

Useful when OCR results look wrong: this is the exact bitmap the OCR engine was handed. Set `SHOW_PAGE_IMAGES = True` in the config cell to enable.

In [ ]:
if SHOW_PAGE_IMAGES:
    for page_no, page in doc.pages.items():
        if page.image is None or page.image.pil_image is None:
            continue
        print(f"--- page {page_no}")
        display(page.image.pil_image)
else:
    print("SHOW_PAGE_IMAGES is False, skipping page renders.")

## Save the outputs

Writes to `output/docling/<pdf-stem>/`:

- `<stem>-embedded.md`: Markdown with images inlined as base64, a single portable file
- `<stem>-refs.md` and `<stem>.html`: Markdown/HTML referencing PNGs in the same folder
- `<stem>.json`: the full DoclingDocument, the lossless representation to feed downstream
- `page-*.png`, `picture-*.png`, `table-*.png`: the extracted bitmaps

In [ ]:
out_dir = OUTPUT_DIR / doc_stem
out_dir.mkdir(parents=True, exist_ok=True)

written: list[Path] = []

# Page bitmaps
for page_no, page in doc.pages.items():
    if page.image is None or page.image.pil_image is None:
        continue
    path = out_dir / f"page-{page_no:03d}.png"
    page.image.pil_image.save(path, "PNG")
    written.append(path)

# Figure and table crops
for i, picture in enumerate(pictures, start=1):
    image = picture.get_image(doc)
    if image is None:
        continue
    path = out_dir / f"picture-{i:03d}.png"
    image.save(path, "PNG")
    written.append(path)

for i, table in enumerate(tables, start=1):
    image = table.get_image(doc)
    if image is None:
        continue
    path = out_dir / f"table-{i:03d}.png"
    image.save(path, "PNG")
    written.append(path)

# Tables as CSV, handy for a quick sanity check in Excel
for i, df in enumerate(dataframes, start=1):
    path = out_dir / f"table-{i:03d}.csv"
    df.to_csv(path, index=False, encoding="utf-8")
    written.append(path)

# Documents
md_embedded = out_dir / f"{doc_stem}-embedded.md"
doc.save_as_markdown(md_embedded, image_mode=ImageRefMode.EMBEDDED)
written.append(md_embedded)

md_refs = out_dir / f"{doc_stem}-refs.md"
doc.save_as_markdown(md_refs, image_mode=ImageRefMode.REFERENCED)
written.append(md_refs)

html_path = out_dir / f"{doc_stem}.html"
doc.save_as_html(html_path, image_mode=ImageRefMode.REFERENCED)
written.append(html_path)

json_path = out_dir / f"{doc_stem}.json"
doc.save_as_json(json_path)
written.append(json_path)

print(f"wrote {len(written)} files to {out_dir}\n")
for path in written[:40]:
    print(f"  {path.name}")
if len(written) > 40:
    print(f"  ... and {len(written) - 40} more")

## Batch over the fixtures (optional)

`convert_all` streams documents through the same converter and keeps going when one fails. Set `RUN_BATCH = True` to process every fixture PDF.

In [ ]:
RUN_BATCH = False

if RUN_BATCH:
    start = time.perf_counter()
    for res in converter.convert_all(_fixtures, raises_on_error=False):
        name = res.input.file.name
        if res.document is None:
            print(f"{name}: FAILED ({res.status})")
            continue
        n_tables = sum(1 for el, _ in res.document.iterate_items() if isinstance(el, TableItem))
        n_pics = sum(1 for el, _ in res.document.iterate_items() if isinstance(el, PictureItem))
        md = res.document.export_to_markdown()
        print(
            f"{name}: {res.status} | pages={len(res.document.pages)} "
            f"tables={n_tables} pictures={n_pics} markdown={len(md):,} chars"
        )
    print(f"\nbatch elapsed: {time.perf_counter() - start:.1f}s")
else:
    print("RUN_BATCH is False, skipping.")

## Tuning notes

**Scanned PDFs with no text layer.** Set `FORCE_FULL_PAGE_OCR = True` in the config cell and rerun `build_pdf_converter()`. Every page then goes through OCR rather than only the bitmap regions.

**Swapping OCR engine.** `EasyOcrOptions` is the default and needs no system install. Alternatives, all from `docling.datamodel.pipeline_options`:

```python
from docling.datamodel.pipeline_options import (
    OcrMode,
    RapidOcrOptions,        # ONNX based, no system dependency, fast on CPU
    TesseractCliOcrOptions, # shells out to the tesseract binary
    TesseractOcrOptions,    # via tesserocr bindings
)

pipeline_options.ocr_options = TesseractCliOcrOptions(mode=OcrMode.FULL_PAGE)
```

Tesseract needs the binary on PATH (`winget install UB-Mannheim.TesseractOCR` on Windows) and `TESSDATA_PREFIX` pointing at its `tessdata` folder.

**Speed.** `TableFormerMode.FAST` cuts table time noticeably. Dropping `images_scale` to `1.0` and setting `generate_page_images = False` saves memory, but then table crops are unavailable. `PAGE_RANGE = (1, 5)` is the quickest way to iterate.

**Downstream use.** Prefer the JSON `DoclingDocument` over the Markdown when feeding another system: it keeps bounding boxes, page provenance and the table cell grid, all of which Markdown flattens away. For RAG, `docling.chunking.HybridChunker` chunks a `DoclingDocument` while respecting that structure.